# CrewAI 101: Building a Multi-Agent Workflow

This notebook builds a current CrewAI workflow with:

- OpenAI as the LLM provider
- `.env` loading with `python-dotenv`
- Tavily web search through `TavilySearchTool`
- a sequential crew with research, writing, and social-media tasks
- completed exercises at the end of the lab

## Setup

Dependencies are managed by the repository environment. The notebook expects environment variables to be available in `.env`.

Required:

```env
OPENAI_API_KEY=...
```

For Tavily search, CrewAI's `TavilySearchTool` expects a Tavily key in most environments:

```env
TAVILY_API_KEY=...
```

Tavily may offer a free tier or trial quota, but the tool still normally authenticates with an API key.

In [1]:
from dotenv import load_dotenv
from crewai import Agent, Crew, LLM, Process, Task
from crewai_tools import TavilySearchTool

load_dotenv()

True

In [ ]:
# SWITCH OFF CrewAI's OpenTelemetry (OTEL) data gathering!
import os
os.environ["OTEL_SDK_DISABLED"] = "true"

In [12]:
llm = LLM(
    model="openai/gpt-4o",
    temperature=0.2,
)

search_tool = TavilySearchTool()

## What CrewAI Builds

CrewAI organizes work around four main objects:

- `Agent`: the role/persona doing the work
- `Task`: the assignment given to an agent
- `Tool`: optional capabilities such as web search
- `Crew`: the team plus execution process

This lab creates a content pipeline:

1. A research analyst gathers current information with Tavily.
2. A writer turns the research into a polished article.
3. A social media strategist turns the article into short promotional posts.

## Agents

The research agent gets the Tavily search tool. The writer and social strategist use the OpenAI model but do not need web tools because they work from previous task context.

In [13]:
research_agent = Agent(
    role="Senior Research Analyst",
    goal="Find current, accurate, source-aware insights about {topic}",
    backstory=(
        "You are an experienced technology researcher. You identify relevant "
        "developments, separate signal from hype, and summarize findings clearly."
    ),
    llm=llm,
    tools=[search_tool],
    verbose=True,
    allow_delegation=False,
)

writer_agent = Agent(
    role="Technology Content Strategist",
    goal="Turn research findings into clear, useful, engaging long-form content",
    backstory=(
        "You are a technical content strategist who explains complex topics for "
        "business and engineering audiences without losing important nuance."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

social_agent = Agent(
    role="Social Media Strategist",
    goal="Create concise platform-ready posts that amplify the article's main ideas",
    backstory=(
        "You are a digital storyteller who turns long-form technical content into "
        "clear, engaging LinkedIn and X/Twitter posts."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

## Tasks

Tasks describe the work and expected output. `context=[previous_task]` makes the dependency between sequential tasks explicit.

In [14]:
research_task = Task(
    description=(
        "Research the latest important developments about {topic}. Use Tavily "
        "search to find current information. Focus on concrete examples, trends, "
        "and practical implications."
    ),
    expected_output=(
        "A concise research brief with key findings, recent examples, relevant "
        "source-aware observations, and practical implications."
    ),
    agent=research_agent,
)

writer_task = Task(
    description=(
        "Using the research brief, write a polished short article about {topic}. "
        "Make it clear, structured, and useful for a technical business audience."
    ),
    expected_output=(
        "A well-structured article with a clear title, short introduction, 3-5 "
        "key sections, and a concise conclusion."
    ),
    agent=writer_agent,
    context=[research_task],
)

social_task = Task(
    description=(
        "Using the final article, create platform-ready social media content about {topic}. "
        "Include 2 LinkedIn post options and 3 short X/Twitter-style posts."
    ),
    expected_output=(
        "Two LinkedIn posts and three short X/Twitter posts with clear hooks, "
        "practical takeaways, and no unsupported claims."
    ),
    agent=social_agent,
    context=[writer_task],
)

## Crew Workflow

`Process.sequential` runs tasks in order: research, writing, then social content. The final crew output comes from the last task, while `tasks_output` keeps each task result for inspection.

In [15]:
content_crew = Crew(
    agents=[research_agent, writer_agent, social_agent],
    tasks=[research_task, writer_task, social_task],
    process=Process.sequential,
    verbose=True,
)

In [16]:
result = content_crew.kickoff(
    inputs={"topic": "latest generative AI breakthroughs"}
)

print("Final output:\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f432d6d8-4ab1-40b2-9cb8-182385b4a158                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the latest important developments about latest generative AI breakthroughs. Use Tavily search   │
│  to find current information. Focus on concrete examples, trends, and practical implications.                   │
│  ID: 16474915-46ee-4214-bf67-2c2610b0cedb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Research the latest important developments about latest generative AI breakthroughs. Use Tavily search   │
│  to find current information. Focus on concrete examples, trends, and practical implications.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#8) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {'query': 'latest generative AI breakthroughs 2023'}                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool tavily_search executed with result: {
  "query": "latest generative AI breakthroughs 2023",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.websensa.com/blog/13-breakthroug...

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Social Media Strategist                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **LinkedIn Post Option 1:**                                                                                    │
│                                                                                                                 │
│  🚀 **Generative AI in 2023: A Game-Changer for Business and Education** 🚀                                     │
│                                                                                                                 │
│  The generative AI landscape is evolving at breakneck speed, with 2023 marking a year of groundbreaking         │
│  advancements. From Meta’s LLaMA 2 to OpenAI’s GPT-4, these models are setting new standards in AI              │
│  capabilities, transforming customer interactions and automating complex tasks.                                 │
│                                                                                                                 │
│  For businesses, this means a shift towards hyper-personalization and enhanced customer experiences, powered    │
│  by predictive analytics. In education, AI is revolutionizing teaching methods, necessitating new policies and  │
│  practices.                                                                                                     │
│                                                                                                                 │
│  However, with great power comes great responsibility. Ethical considerations and AI literacy are crucial as    │
│  we integrate these technologies into our daily lives.                                                          │
│                                                                                                                 │
│  Are you ready to harness the potential of generative AI? Let's discuss how your organization can stay ahead    │
│  in this AI revolution. #GenerativeAI #AIRevolution #BusinessInnovation                                         │
│                                                                                                                 │
│  **LinkedIn Post Option 2:**                                                                                    │
│                                                                                                                 │
│  🌟 **Navigating the Generative AI Revolution: Breakthroughs & Implications** 🌟                                │
│                                                                                                                 │
│  2023 has been a landmark year for generative AI, with innovations like Google’s Bard and Stanford's DetectGPT  │
│  pushing the boundaries of what's possible. These advancements are not just technical feats; they're reshaping  │
│  industries and redefining possibilities.                                                                       │
│                                                                                                                 │
│  Key trends include the rise of transformers in AI applications, driving hyper-personalization and              │
│  transforming customer engagement. In education, AI is a catalyst for change, prompting a reevaluation of       │
│  teaching practices and policies.                                                                               │
│                                                            

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the final article, create platform-ready social media content about latest generative AI           │
│  breakthroughs. Include 2 LinkedIn post options and 3 short X/Twitter-style posts.                              │
│  Agent: Social Media Strategist                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final output:

**LinkedIn Post Option 1:**

🚀 **Generative AI in 2023: A Game-Changer for Business and Education** 🚀

The generative AI landscape is evolving at breakneck speed, with 2023 marking a year of groundbreaking advancements. From Meta’s LLaMA 2 to OpenAI’s GPT-4, these models are setting new standards in AI capabilities, transforming customer interactions and automating complex tasks. 

For businesses, this means a shift towards hyper-personalization and enhanced customer experiences, powered by predictive analytics. In education, AI is revolutionizing teaching methods, necessitating new policies and practices. 

However, with great power comes great responsibility. Ethical considerations and AI literacy are crucial as we integrate these technologies into our daily lives. 

Are you ready to harness the potential of generative AI? Let's discuss how your organization can stay ahead in this AI revolution. #GenerativeAI #AIRevolution #BusinessInnovation

**LinkedIn Post Option 2:

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f432d6d8-4ab1-40b2-9cb8-182385b4a158                                                                       │
│  Final Output: **LinkedIn Post Option 1:**                                                                      │
│                                                                                                                 │
│  🚀 **Generative AI in 2023: A Game-Changer for Business and Education** 🚀                                     │
│                                                                                                                 │
│  The generative AI landscape is evolving at breakneck speed, with 2023 marking a year of groundbreaking         │
│  advancements. From Meta’s LLaMA 2 to OpenAI’s GPT-4, these models are setting new standards in AI              │
│  capabilities, transforming customer interactions and automating complex tasks.                                 │
│                                                                                                                 │
│  For businesses, this means a shift towards hyper-personalization and enhanced customer experiences, powered    │
│  by predictive analytics. In education, AI is revolutionizing teaching methods, necessitating new policies and  │
│  practices.                                                                                                     │
│                                                                                                                 │
│  However, with great power comes great responsibility. Ethical considerations and AI literacy are crucial as    │
│  we integrate these technologies into our daily lives.                                                          │
│                                                                                                                 │
│  Are you ready to harness the potential of generative AI? Let's discuss how your organization can stay ahead    │
│  in this AI revolution. #GenerativeAI #AIRevolution #BusinessInnovation                                         │
│                                                                                                                 │
│  **LinkedIn Post Option 2:**                                                                                    │
│                                                                                                                 │
│  🌟 **Navigating the Generative AI Revolution: Breakthroughs & Implications** 🌟                                │
│                                                                                                                 │
│  2023 has been a landmark year for generative AI, with innovations like Google’s Bard and Stanford's DetectGPT  │
│  pushing the boundaries of what's possible. These advancements are not just technical feats; they're reshaping  │
│  industries and redefining possibilities.                                                                       │
│                                                                                                                 │
│  Key trends include the rise of transformers in AI applications, driving hyper-personalization and              │
│  transforming customer engagement. In education, AI is a catalyst for change, prompting a reevaluation of       │
│  teaching practices and policies.                                                                               │
│                                                           

## Inspect Results

CrewAI returns a result object. The most useful fields are:

- `result.raw`: the final output from the crew
- `result.tasks_output`: individual task outputs
- `result.token_usage`: token accounting when available

In [17]:
print("Task outputs:\n")
for index, task_output in enumerate(result.tasks_output, start=1):
    print(f"Task {index}: {task_output.description}")
    print(task_output.raw)
    print("-" * 80)

print("Token usage:")
print(result.token_usage)

Task outputs:

Task 1: Research the latest important developments about latest generative AI breakthroughs. Use Tavily search to find current information. Focus on concrete examples, trends, and practical implications.
**Research Brief: Latest Generative AI Breakthroughs in 2023**

**Key Findings:**
1. **Concrete Examples:**
   - The rise of large language models like ChatGPT has been a significant development, showcasing the explosion in popularity and application range of generative AI (Turrentine Brokerage).
   - Meta’s LLaMA 2, Google’s Bard chatbot, and OpenAI’s GPT-4 are among the notable AI models launched in 2023 (MIT Technology Review).
   - Stanford developed DetectGPT, a tool to distinguish between human and AI-generated text, highlighting advancements in AI detection (Stanford HAI).

2. **Trends:**
   - The use of transformers, a type of neural network, has become prevalent for tasks ranging from text and image generation to more complex applications (Forbes).
   - Hyper-pe

## Exercises

The original lab exercises are completed below. They add a social media publishing layer to the CrewAI workflow.

### Completed Exercise 1: Create a Social Media Strategist Agent

Create a `Social Media Strategist` agent that can turn the final article into platform-ready short-form content.

In [18]:
exercise_social_agent = Agent(
    role="Social Media Strategist",
    goal="Generate engaging social media snippets based on the final article about {topic}",
    backstory=(
        "You are a digital storyteller who turns long-form technical content into "
        "clear, engaging LinkedIn and X/Twitter posts that drive readers back to "
        "the full article."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

### Completed Exercise 2: Define a Social Media Strategy Task

Create a task that uses the writer task as context and asks the social media strategist to produce platform-specific posts.

In [19]:
exercise_social_task = Task(
    description=(
        "Summarize the article about {topic} into platform-ready social media content. "
        "Create 2 LinkedIn post options and 3 concise X/Twitter-style posts. "
        "Keep the tone professional, useful, and grounded in the article."
    ),
    expected_output=(
        "Two LinkedIn post options and three X/Twitter-style posts with clear hooks, "
        "practical takeaways, and no unsupported claims."
    ),
    agent=exercise_social_agent,
    context=[writer_task],
)

### Completed Exercise 3: Create and Run the Complete Crew

Assemble a sequential crew with research, writing, and social media tasks, then run it with `kickoff(inputs={...})`.

In [20]:
exercise_crew = Crew(
    agents=[research_agent, writer_agent, exercise_social_agent],
    tasks=[research_task, writer_task, exercise_social_task],
    process=Process.sequential,
    verbose=True,
)

exercise_result = exercise_crew.kickoff(
    inputs={"topic": "latest generative AI breakthroughs"}
)

print("Exercise final output:\n")
print(exercise_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d07f696d-0639-4d64-8903-b76d72884583                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the latest important developments about latest generative AI breakthroughs. Use Tavily search   │
│  to find current information. Focus on concrete examples, trends, and practical implications.                   │
│  ID: 16474915-46ee-4214-bf67-2c2610b0cedb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Research the latest important developments about latest generative AI breakthroughs. Use Tavily search   │
│  to find current information. Focus on concrete examples, trends, and practical implications.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#12) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {'query': 'latest generative AI breakthroughs 2023'}                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool tavily_search executed with result: {
  "query": "latest generative AI breakthroughs 2023",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.websensa.com/blog/13-breakthroug...

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Social Media Strategist                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **LinkedIn Post Option 1:**                                                                                    │
│                                                                                                                 │
│  🌟 **Navigating the Generative AI Revolution: Key Breakthroughs & Implications** 🌟                            │
│                                                                                                                 │
│  2023 has been a landmark year for generative AI, with rapid growth and adoption reshaping industries. 🚀       │
│  According to McKinsey, 1/3 of businesses are already using AI tools, and 40% plan to boost their AI            │
│  investments. This isn't just a trend—it's a transformative shift.                                              │
│                                                                                                                 │
│  Key breakthroughs like OpenAI's ChatGPT and Salesforce's Einstein GPT are revolutionizing text generation and  │
│  CRM systems, respectively. These advancements are not only enhancing efficiency but also redefining how        │
│  businesses interact with customers.                                                                            │
│                                                                                                                 │
│  Generative AI is actively transforming sectors like marketing, sales, and product development, unlocking new   │
│  levels of productivity and creativity. However, with great power comes great responsibility. Ethical           │
│  considerations and regulatory frameworks are crucial to ensure AI's positive impact.                           │
│                                                                                                                 │
│  As we embrace this AI-driven future, businesses must invest in technology and workforce reskilling to stay     │
│  competitive. Are you ready to lead the charge in the generative AI revolution? 🌐                              │
│                                                                                                                 │
│  #GenerativeAI #Innovation #BusinessTransformation #AIRevolution                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **LinkedIn Post Option 2:**                                                                                    │
│                                                                                                                 │
│  🔍 **Exploring the Impact of Generative AI: Breakthroughs, Applications, and Challenges** 🔍                   │
│                                                                                                                 │
│  The generative AI landscape is evolving at an unprecedented pace in 2023. With tools like ChatGPT reaching     │
│  100 million users in just two months, the potential for AI in natural language processing is undeniable.       │
│  Similarly, Salesforce's Einstein GPT is setting new standard

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Summarize the article about latest generative AI breakthroughs into platform-ready social media          │
│  content. Create 2 LinkedIn post options and 3 concise X/Twitter-style posts. Keep the tone professional,       │
│  useful, and grounded in the article.                                                                           │
│  Agent: Social Media Strategist                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exercise final output:

**LinkedIn Post Option 1:**

🌟 **Navigating the Generative AI Revolution: Key Breakthroughs & Implications** 🌟

2023 has been a landmark year for generative AI, with rapid growth and adoption reshaping industries. 🚀 According to McKinsey, 1/3 of businesses are already using AI tools, and 40% plan to boost their AI investments. This isn't just a trend—it's a transformative shift.

Key breakthroughs like OpenAI's ChatGPT and Salesforce's Einstein GPT are revolutionizing text generation and CRM systems, respectively. These advancements are not only enhancing efficiency but also redefining how businesses interact with customers.

Generative AI is actively transforming sectors like marketing, sales, and product development, unlocking new levels of productivity and creativity. However, with great power comes great responsibility. Ethical considerations and regulatory frameworks are crucial to ensure AI's positive impact.

As we embrace this AI-driven future, businesse

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d07f696d-0639-4d64-8903-b76d72884583                                                                       │
│  Final Output: **LinkedIn Post Option 1:**                                                                      │
│                                                                                                                 │
│  🌟 **Navigating the Generative AI Revolution: Key Breakthroughs & Implications** 🌟                            │
│                                                                                                                 │
│  2023 has been a landmark year for generative AI, with rapid growth and adoption reshaping industries. 🚀       │
│  According to McKinsey, 1/3 of businesses are already using AI tools, and 40% plan to boost their AI            │
│  investments. This isn't just a trend—it's a transformative shift.                                              │
│                                                                                                                 │
│  Key breakthroughs like OpenAI's ChatGPT and Salesforce's Einstein GPT are revolutionizing text generation and  │
│  CRM systems, respectively. These advancements are not only enhancing efficiency but also redefining how        │
│  businesses interact with customers.                                                                            │
│                                                                                                                 │
│  Generative AI is actively transforming sectors like marketing, sales, and product development, unlocking new   │
│  levels of productivity and creativity. However, with great power comes great responsibility. Ethical           │
│  considerations and regulatory frameworks are crucial to ensure AI's positive impact.                           │
│                                                                                                                 │
│  As we embrace this AI-driven future, businesses must invest in technology and workforce reskilling to stay     │
│  competitive. Are you ready to lead the charge in the generative AI revolution? 🌐                              │
│                                                                                                                 │
│  #GenerativeAI #Innovation #BusinessTransformation #AIRevolution                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **LinkedIn Post Option 2:**                                                                                    │
│                                                                                                                 │
│  🔍 **Exploring the Impact of Generative AI: Breakthroughs, Applications, and Challenges** 🔍                   │
│                                                                                                                 │
│  The generative AI landscape is evolving at an unprecedented pace in 2023. With tools like ChatGPT reaching     │
│  100 million users in just two months, the potential for AI in natural language processing is undeniable.       │
│  Similarly, Salesforce's Einstein GPT is setting new standar

### Exercise Solution Notes

The main workflow earlier in the notebook already uses the same completed social-media agent and task. These exercise cells show the requested components explicitly, using separate `exercise_*` names so they can be run independently without overwriting the main objects.

## Authors

[Karan Goswami](https://author.skills.network/instructors/karan_goswami)

[Kunal Makwana](https://author.skills.network/instructors/kunal_makwana)
